###### Notebook 1 — 03_Load_Silver_to_Gold_Augment.py

In [0]:
%run "../../utils/metrics_query"


In [0]:
mq = MetricsQuery(spark)
print(":white_check_mark: MetricsQuery loaded")

In [0]:
%run "../../utils/cdc_validator"


In [0]:
%run "../../utils/audit_logger"

In [0]:

#NOTEBOOK 1 WITH 3MODULES applied from utils folder(03_Load_Silver_to_Gold_Augment.py)

from pyspark.sql.functions import col, lit, lower, when, current_timestamp, coalesce, split
from delta.tables import DeltaTable
import time

# ===================================================================================
# CONFIGURATION
# ===================================================================================
dbutils.widgets.text("customer_id", "cust_001")
customer_id = dbutils.widgets.get("customer_id").lower()

load_type = "incremental"  # hardcoded — change to "historical" if full backfill needed

# ===================================================================================
# FUTURE ENHANCEMENT — Multi-Customer Loop (uncomment when ready):
#
# customer_ids_df = spark.sql("""
#     SELECT DISTINCT customer_id
#     FROM ingestion_metadata.customers
#     WHERE is_active = true
# """)
# customer_ids = [row.customer_id for row in customer_ids_df.collect()]
# for customer_id in customer_ids:
#     pass
# ===================================================================================

SILVER_CATALOG     = "hgi"
SILVER_SCHEMA      = "silver"
GOLD_CATALOG       = "hgi"
GOLD_SCHEMA        = "gold"
ENRICHMENT_CATALOG = "hgi"
ENRICHMENT_SCHEMA  = "enrichment"
STALE_DAYS         = 180

FREE_EMAIL_DOMAINS = [
    'gmail.com', 'yahoo.com', 'hotmail.com',
    'aol.com', 'outlook.com', 'icloud.com'
]

spark.conf.set("spark.sql.shuffle.partitions", 4)

print("=" * 70)
print("03_Load_Silver_to_Gold_Augment")
print(f"  Customer : {customer_id}")
print(f"  Load Type: {load_type}")
print("=" * 70)

# CDF_CHECK — list to accumulate all counts, written to table at the end
cdf_counts = []  # CDF_CHECK

# CDF_CHECK — helper to record a count entry
def record_count(stage, table_name, count):          # CDF_CHECK
    cdf_counts.append({                              # CDF_CHECK
        "stage"      : stage,                        # CDF_CHECK
        "table_name" : table_name,                   # CDF_CHECK
        "row_count"  : count,                        # CDF_CHECK
        "customer_id": customer_id                   # CDF_CHECK
    })                                               # CDF_CHECK


# ===================================================================================
# MODULE 1 — METRICS QUERY (START)
# ===================================================================================

#%run ../../utils/metrics_query

#print("✅ MetricsQuery loaded")


# ===================================================================================

spark.sql("ALTER TABLE hgi.silver.contacts_all SET TBLPROPERTIES ('delta.enableChangeDataFeed' = 'true')")
spark.sql("ALTER TABLE hgi.silver.accounts_all SET TBLPROPERTIES ('delta.enableChangeDataFeed' = 'true')")
spark.sql("ALTER TABLE hgi.silver.contacts SET TBLPROPERTIES ('delta.enableChangeDataFeed' = 'true')")
spark.sql("ALTER TABLE hgi.silver.events SET TBLPROPERTIES ('delta.enableChangeDataFeed' = 'true')")






#=============================================================================
# MODULE 2 — CDC VALIDATOR (START)
# Validates CDF is enabled on silver input tables before processing.
# Writes validation results to hgi.silver.logs.cdc_validation_log
# ===================================================================================
#%run "../../utils/cdc_validator"

cdc_silver_result = validate_cdc(
    layer       = "silver",
    table_names = ["contacts_all", "contacts", "events", "accounts_all"],
    catalog     = SILVER_CATALOG,
    schema      = SILVER_SCHEMA
)
print(f"  CDC Silver: {cdc_silver_result['message']}")


# ===================================================================================
# MODULE 3 — AUDIT LOGGER (START)
# ===================================================================================
#%run "../../utils/audit_logger"
initialize_audit_tables()
print("✅ AuditLogger loaded and tables initialized")

# ===================================================================================
# CDF_CHECK — CREATE cdf_checker TABLE
# ===================================================================================
spark.sql("CREATE SCHEMA IF NOT EXISTS bronze.cdf_checks")  # CDF_CHECK

spark.sql("""                                               
    CREATE TABLE IF NOT EXISTS bronze.cdf_checks.cdf_checker (
        run_timestamp TIMESTAMP,
        customer_id   STRING,
        stage         STRING,
        table_name    STRING,
        row_count     LONG
    ) USING DELTA
""")                                                        # CDF_CHECK
print("✅ [CDF_CHECK] bronze.cdf_checks.cdf_checker ready") # CDF_CHECK


# ===================================================================================
# GOLD TABLE INITIALIZATION
# ===================================================================================
spark.sql(f"CREATE SCHEMA IF NOT EXISTS {GOLD_CATALOG}.{GOLD_SCHEMA}")

def initialize_gold_tables():
    spark.sql(f"""
        CREATE TABLE IF NOT EXISTS {GOLD_CATALOG}.{GOLD_SCHEMA}.persons (
            mk_email              STRING    COMMENT 'Lookup key lowercase email',
            mk_domain             STRING,
            mk_status             STRING,
            mk_timestamp          TIMESTAMP,
            name__fullname        STRING,
            name__givenname       STRING,
            name__familyname      STRING,
            employment__name      STRING,
            employment__domain    STRING,
            employment__role      STRING,
            employment__seniority STRING,
            employment__title     STRING,
            geo__city             STRING,
            geo__country          STRING,
            geo__state            STRING,
            is_student            BOOLEAN,
            is_spam               BOOLEAN,
            linkedin__handle      STRING,
            twitter__handle       STRING,
            updated_at            TIMESTAMP
        ) USING DELTA CLUSTER BY (mk_email)
        TBLPROPERTIES (
            'delta.enableDeletionVectors'      = 'true',
            'delta.autoOptimize.optimizeWrite' = 'true',
            'delta.autoOptimize.autoCompact'   = 'true'
        )
    """)

    spark.sql(f"""
        CREATE TABLE IF NOT EXISTS {GOLD_CATALOG}.{GOLD_SCHEMA}.companies (
            mk_domain               STRING    COMMENT 'Lookup key lowercase domain',
            mk_status               STRING,
            mk_timestamp            TIMESTAMP,
            name                    STRING,
            description             STRING,
            category__industry      STRING,
            category__sector        STRING,
            category__industrygroup STRING,
            category__subindustry   STRING,
            metrics__employees      LONG,
            metrics__employeesrange STRING,
            metrics__raised         LONG,
            metrics__marketcap      STRING,
            geo__city               STRING,
            geo__country            STRING,
            geo__state              STRING,
            tech                    STRING,
            tags                    STRING,
            founded_year            LONG,
            emailprovider           BOOLEAN,
            site_url                STRING,
            alexa__globalrank       LONG,
            is_personal             BOOLEAN,
            updated_at              TIMESTAMP
        ) USING DELTA CLUSTER BY (mk_domain)
        TBLPROPERTIES (
            'delta.enableDeletionVectors'      = 'true',
            'delta.autoOptimize.optimizeWrite' = 'true',
            'delta.autoOptimize.autoCompact'   = 'true'
        )
    """)

    spark.sql(f"""
        CREATE TABLE IF NOT EXISTS {GOLD_CATALOG}.{GOLD_SCHEMA}.contacts_computations (
            contact_id                        STRING,
            tenant                           LONG,
            email                            STRING,
            domain                           STRING,
            person__mk_email                 STRING,
            person__mk_domain                STRING,
            person__mk_status                STRING,
            person__mk_timestamp             TIMESTAMP,
            person__name__fullname           STRING,
            person__name__givenname          STRING,
            person__name__familyname         STRING,
            person__employment__name         STRING,
            person__employment__domain       STRING,
            person__employment__role         STRING,
            person__employment__seniority    STRING,
            person__employment__title        STRING,
            person__geo__city                STRING,
            person__geo__country             STRING,
            person__geo__state               STRING,
            person__is_student               BOOLEAN,
            person__is_spam                  BOOLEAN,
            person__linkedin__handle         STRING,
            person__twitter__handle          STRING,
            company__mk_domain               STRING,
            company__mk_status               STRING,
            company__mk_timestamp            TIMESTAMP,
            company__name                    STRING,
            company__description             STRING,
            company__category__industry      STRING,
            company__category__sector        STRING,
            company__category__industrygroup STRING,
            company__category__subindustry   STRING,
            company__metrics__employees      LONG,
            company__metrics__employeesrange STRING,
            company__metrics__raised         LONG,
            company__metrics__marketcap      STRING,
            company__geo__city               STRING,
            company__geo__country            STRING,
            company__geo__state              STRING,
            company__tech                    STRING,
            company__tags                    STRING,
            company__founded_year            LONG,
            company__emailprovider           BOOLEAN,
            company__site_url                STRING,
            company__alexa__globalrank       LONG,
            company__is_personal             BOOLEAN,
            updated_at                       TIMESTAMP
        ) USING DELTA CLUSTER BY (contact_id)
        TBLPROPERTIES (
            'delta.enableDeletionVectors'      = 'true',
            'delta.autoOptimize.optimizeWrite' = 'true',
            'delta.autoOptimize.autoCompact'   = 'true'
        )
    """)

initialize_gold_tables()
print("Gold tables initialized.")

# ===================================================================================
# WATERMARK HELPERS
# ===================================================================================
def get_watermark(src_sys, obj_name):
    try:
        result = spark.sql(f"""
            SELECT last_processed_timestamp
            FROM ingestion_metadata.watermarks
            WHERE source_system = '{src_sys}' AND object_name = '{obj_name}'
        """).collect()
        if result:
            return result[0]["last_processed_timestamp"]
    except Exception:
        pass
    return None

def update_watermark(src_sys, obj_name):
    spark.sql(f"""
        MERGE INTO ingestion_metadata.watermarks AS target
        USING (SELECT '{src_sys}' AS source_system, '{obj_name}' AS object_name,
                      current_timestamp() AS last_processed_timestamp) AS source
        ON target.source_system = source.source_system
        AND target.object_name  = source.object_name
        WHEN MATCHED THEN UPDATE SET
            target.last_processed_timestamp = source.last_processed_timestamp
        WHEN NOT MATCHED THEN INSERT
            (source_system, object_name, last_processed_timestamp)
            VALUES (source.source_system, source.object_name, source.last_processed_timestamp)
    """)
    print(f"  Watermark updated: source_system='{src_sys}', object_name='{obj_name}'")


# ===================================================================================
# MAIN PIPELINE
# ===================================================================================
start_time               = time.time()
status                   = "success"
error_type               = None
error_reason             = None
email_count              = 0
domain_count             = 0
persons_enriched_count   = 0
companies_enriched_count = 0
compute_count            = 0
cdf_validation_status    = "NOT_CHECKED"
total_rows               = 0

try:
    # CDF_CHECK — silver input table counts before processing
    for tbl in ["contacts_all", "contacts", "events", "accounts_all"]:                           # CDF_CHECK
        _ct = spark.sql(f"SELECT COUNT(*) AS cnt FROM {SILVER_CATALOG}.{SILVER_SCHEMA}.{tbl}").collect()[0]["cnt"]  # CDF_CHECK
        print(f"  [CDF_CHECK] silver.{tbl} = {_ct}")                                             # CDF_CHECK
        record_count(f"silver_{tbl}", f"{SILVER_CATALOG}.{SILVER_SCHEMA}.{tbl}", _ct)            # CDF_CHECK
    
    # -----------------------------------------------------------------------
    # STEP 0A — EMAIL CANDIDATES
    # -----------------------------------------------------------------------
    print("\n" + "="*70)
    print("STEP 0A — EMAIL CANDIDATES DATAFRAME")
    print("="*70)

    if load_type == "historical":
        df_email_candidates = spark.sql(f"""
            SELECT DISTINCT lower(email) AS email
            FROM {SILVER_CATALOG}.{SILVER_SCHEMA}.contacts_all
            WHERE email IS NOT NULL
        """)
    else:
        df_email_candidates = spark.sql(f"""
            WITH candidates AS (
                SELECT DISTINCT lower(ca.email) AS email, 1 AS priority
                FROM {SILVER_CATALOG}.{SILVER_SCHEMA}.contacts_all ca
                JOIN {SILVER_CATALOG}.{SILVER_SCHEMA}.contacts uc ON ca.id = uc.id
                WHERE ca.email IS NOT NULL
                  --AND uc.data_timestamp >= date_add(current_date(), -7)
                UNION
                SELECT DISTINCT lower(ca.email) AS email, 2 AS priority
                FROM {SILVER_CATALOG}.{SILVER_SCHEMA}.contacts_all ca
                JOIN {SILVER_CATALOG}.{SILVER_SCHEMA}.events e ON ca.id = e.contact_id
                WHERE ca.email IS NOT NULL
                  -- AND e.meta_event IN ('signup', 'conversion')
                  AND e.event_timestamp >= date_add(current_date(), -30)
                UNION
                SELECT DISTINCT lower(ca.email) AS email, 3 AS priority
                FROM {SILVER_CATALOG}.{SILVER_SCHEMA}.contacts_all ca
                JOIN {SILVER_CATALOG}.{SILVER_SCHEMA}.events e ON ca.id = e.contact_id
                WHERE ca.email IS NOT NULL
                  AND e.event_timestamp >= date_add(current_date(), -7)
            ),
            ranked AS (
                SELECT email, MIN(priority) AS priority
                FROM candidates GROUP BY email
            )
            SELECT r.email
            FROM ranked r
            LEFT JOIN {GOLD_CATALOG}.{GOLD_SCHEMA}.persons p ON r.email = lower(p.mk_email)
            WHERE p.mk_email IS NULL
               OR p.mk_status IN ('error', 'not_found')
               OR (p.mk_timestamp IS NOT NULL AND p.mk_timestamp < date_add(current_date(), -{STALE_DAYS}))
            ORDER BY r.priority ASC
        """)

    email_count = df_email_candidates.count()
    print(f"  Email candidates: {email_count}")
    df_email_candidates.show(20, truncate=False)
    
    record_count("0a_email_candidates", f"{SILVER_CATALOG}.{SILVER_SCHEMA}.contacts_all", email_count)  # #CDF_CHECK
    print(f"  [CDF_CHECK] 0A email_candidates = {email_count}")                                          # #CDF_CHECK
    
    
    # -----------------------------------------------------------------------
    # STEP 0B — DOMAIN CANDIDATES
    # -----------------------------------------------------------------------
    print("\n" + "="*70)
    print("STEP 0B — DOMAIN CANDIDATES DATAFRAME")
    print("="*70)

    if load_type == "historical":
        df_domain_candidates = spark.sql(f"""
            SELECT DISTINCT lower(domain) AS domain
            FROM {SILVER_CATALOG}.{SILVER_SCHEMA}.accounts_all
            WHERE domain IS NOT NULL
        """)
    else:
        df_domain_candidates = spark.sql(f"""
            SELECT DISTINCT lower(a.domain) AS domain
            FROM {SILVER_CATALOG}.{SILVER_SCHEMA}.accounts_all a
            LEFT JOIN {GOLD_CATALOG}.{GOLD_SCHEMA}.companies c ON lower(a.domain) = lower(c.mk_domain)
            WHERE a.domain IS NOT NULL
              AND (c.mk_domain IS NULL
                   OR c.mk_status IN ('error', 'not_found')
                   OR(c.mk_timestamp IS NOT NULL AND c.mk_timestamp < date_add(current_date(), -{STALE_DAYS})))
        """)

    domain_count = df_domain_candidates.count()
    print(f"  Domain candidates: {domain_count}")
    df_domain_candidates.show(20, truncate=False)


    record_count("0b_domain_candidates", f"{SILVER_CATALOG}.{SILVER_SCHEMA}.accounts_all", domain_count)  # CDF_CHECK
    print(f"  [CDF_CHECK] 0B domain_candidates = {domain_count}")                                          # CDF_CHECK


    # -----------------------------------------------------------------------
    # STEP 1 — EMAIL ENRICHMENT
    # -----------------------------------------------------------------------
    print("\n" + "="*70)
    print("STEP 1 — EMAIL ENRICHMENT")
    print("="*70)

    df_person_profiles  = spark.table(f"{ENRICHMENT_CATALOG}.{ENRICHMENT_SCHEMA}.person_profiles")
    df_persons_enriched = df_email_candidates.alias("cand").join(
        df_person_profiles.alias("p"),
        lower(col("cand.email")) == lower(col("p.mk_email")), "left"
    ).select(
        coalesce(lower(col("p.mk_email")), lower(col("cand.email"))).alias("mk_email"),
        when(col("p.mk_domain").isNotNull(), col("p.mk_domain"))
            .otherwise(when(col("cand.email").contains("@"),
                            split(lower(col("cand.email")), "@").getItem(1))
                       .otherwise(lit(None))).alias("mk_domain"),
        when(col("p.mk_status").isNotNull(), col("p.mk_status")).otherwise(lit("not_found")).alias("mk_status"),
        when(col("p.mk_timestamp").isNotNull(), col("p.mk_timestamp")).otherwise(current_timestamp()).alias("mk_timestamp"),
        col("p.name__fullname"), col("p.name__givenname"), col("p.name__familyname"),
        col("p.employment__name"), col("p.employment__domain"), col("p.employment__role"),
        col("p.employment__seniority"), col("p.employment__title"),
        col("p.geo__city"), col("p.geo__country"), col("p.geo__state"),
        col("p.is_student"), col("p.is_spam"),
        col("p.linkedin__handle"), col("p.twitter__handle"),
        current_timestamp().alias("updated_at")
    )

    persons_enriched_count = df_persons_enriched.count()
    print(f"  Persons enriched: {persons_enriched_count}")
    df_persons_enriched.show(20, truncate=True)

    DeltaTable.forName(spark, f"{GOLD_CATALOG}.{GOLD_SCHEMA}.persons").alias("target").merge(
        df_persons_enriched.alias("source"),
        "lower(target.mk_email) = lower(source.mk_email)"
    ).whenMatchedUpdate(set={
        "mk_domain":"source.mk_domain","mk_status":"source.mk_status",
        "mk_timestamp":"source.mk_timestamp","name__fullname":"source.name__fullname",
        "name__givenname":"source.name__givenname","name__familyname":"source.name__familyname",
        "employment__name":"source.employment__name","employment__domain":"source.employment__domain",
        "employment__role":"source.employment__role","employment__seniority":"source.employment__seniority",
        "employment__title":"source.employment__title","geo__city":"source.geo__city",
        "geo__country":"source.geo__country","geo__state":"source.geo__state",
        "is_student":"source.is_student","is_spam":"source.is_spam",
        "linkedin__handle":"source.linkedin__handle","twitter__handle":"source.twitter__handle",
        "updated_at":"source.updated_at"
    }).whenNotMatchedInsertAll().execute()

    spark.table(f"{GOLD_CATALOG}.{GOLD_SCHEMA}.persons").show(20, truncate=True)
    print(f"STEP 1 COMPLETE — {persons_enriched_count} persons written to hgi.gold.persons")
    update_watermark("augment", "persons")


    record_count("1_persons_enriched", f"{GOLD_CATALOG}.{GOLD_SCHEMA}.persons", persons_enriched_count)  # CDF_CHECK
    print(f"  [CDF_CHECK] Step 1 persons_enriched = {persons_enriched_count}")                           # CDF_CHECK


    # -----------------------------------------------------------------------
    # STEP 2 — DOMAIN ENRICHMENT
    # -----------------------------------------------------------------------
    print("\n" + "="*70)
    print("STEP 2 — DOMAIN ENRICHMENT")
    print("="*70)

    df_company_profiles   = spark.table(f"{ENRICHMENT_CATALOG}.{ENRICHMENT_SCHEMA}.company_profiles")
    df_companies_enriched = df_domain_candidates.alias("cand").join(
        df_company_profiles.alias("c"),
        lower(col("cand.domain")) == lower(col("c.mk_domain")), "left"
    ).select(
        coalesce(lower(col("c.mk_domain")), lower(col("cand.domain"))).alias("mk_domain"),
        when(col("c.mk_status").isNotNull(), col("c.mk_status")).otherwise(lit("not_found")).alias("mk_status"),
        current_timestamp().alias("mk_timestamp"),
        col("c.name"), col("c.description"),
        col("c.category__industry"), col("c.category__sector"),
        col("c.category__industrygroup"), col("c.category__subindustry"),
        col("c.metrics__employees"), col("c.metrics__employeesrange"),
        col("c.metrics__raised"), col("c.metrics__marketcap"),
        col("c.geo__city"), col("c.geo__country"), col("c.geo__state"),
        col("c.tech"), col("c.tags"), col("c.founded_year"),
        col("c.emailprovider"), col("c.site_url"), col("c.alexa__globalrank"),
        when(lower(col("cand.domain")).isin(FREE_EMAIL_DOMAINS), lit(True))
            .otherwise(lit(False)).alias("is_personal"),
        current_timestamp().alias("updated_at")
    )

    companies_enriched_count = df_companies_enriched.count()
    print(f"  Companies enriched: {companies_enriched_count}")
    df_companies_enriched.show(20, truncate=True)

    DeltaTable.forName(spark, f"{GOLD_CATALOG}.{GOLD_SCHEMA}.companies").alias("target").merge(
        df_companies_enriched.alias("source"),
        "lower(target.mk_domain) = lower(source.mk_domain)"
    ).whenMatchedUpdate(set={
        "mk_status":"source.mk_status","mk_timestamp":"source.mk_timestamp",
        "name":"source.name","description":"source.description",
        "category__industry":"source.category__industry","category__sector":"source.category__sector",
        "category__industrygroup":"source.category__industrygroup",
        "category__subindustry":"source.category__subindustry",
        "metrics__employees":"source.metrics__employees",
        "metrics__employeesrange":"source.metrics__employeesrange",
        "metrics__raised":"source.metrics__raised","metrics__marketcap":"source.metrics__marketcap",
        "geo__city":"source.geo__city","geo__country":"source.geo__country",
        "geo__state":"source.geo__state","tech":"source.tech","tags":"source.tags",
        "founded_year":"source.founded_year","emailprovider":"source.emailprovider",
        "site_url":"source.site_url","alexa__globalrank":"source.alexa__globalrank",
        "is_personal":"source.is_personal","updated_at":"source.updated_at"
    }).whenNotMatchedInsertAll().execute()

    spark.table(f"{GOLD_CATALOG}.{GOLD_SCHEMA}.companies").show(20, truncate=True)
    print(f"STEP 2 COMPLETE — {companies_enriched_count} companies written to hgi.gold.companies")
    update_watermark("augment", "companies")


    record_count("2_companies_enriched", f"{GOLD_CATALOG}.{GOLD_SCHEMA}.companies", companies_enriched_count)  # CDF_CHECK
    print(f"  [CDF_CHECK] Step 2 companies_enriched = {companies_enriched_count}")                             # CDF_CHECK


    # -----------------------------------------------------------------------
    # STEP 3 — COMPUTE (WIDE JOIN) ENHANCED
    # -----------------------------------------------------------------------
    print("\n" + "="*70)
    print("STEP 3 — COMPUTE (WIDE JOIN) ENHANCED")
    print("="*70)

    df_contacts_computations = spark.sql(f"""
        SELECT
            ca.id                                AS contact_id,
            ca.tenant, ca.email, ca.domain,
            p.mk_email                           AS person__mk_email,
            p.mk_domain                          AS person__mk_domain,
            p.mk_status                          AS person__mk_status,
            p.mk_timestamp                       AS person__mk_timestamp,
            p.name__fullname                     AS person__name__fullname,
            p.name__givenname                    AS person__name__givenname,
            p.name__familyname                   AS person__name__familyname,
            p.employment__name                   AS person__employment__name,
            p.employment__domain                 AS person__employment__domain,
            p.employment__role                   AS person__employment__role,
            p.employment__seniority              AS person__employment__seniority,
            p.employment__title                  AS person__employment__title,
            p.geo__city                          AS person__geo__city,
            p.geo__country                       AS person__geo__country,
            p.geo__state                         AS person__geo__state,
            p.is_student                         AS person__is_student,
            p.is_spam                            AS person__is_spam,
            p.linkedin__handle                   AS person__linkedin__handle,
            p.twitter__handle                    AS person__twitter__handle,
            c.mk_domain                          AS company__mk_domain,
            c.mk_status                          AS company__mk_status,
            c.mk_timestamp                       AS company__mk_timestamp,
            c.name                               AS company__name,
            c.description                        AS company__description,
            c.category__industry                 AS company__category__industry,
            c.category__sector                   AS company__category__sector,
            c.category__industrygroup            AS company__category__industrygroup,
            c.category__subindustry              AS company__category__subindustry,
            c.metrics__employees                 AS company__metrics__employees,
            c.metrics__employeesrange            AS company__metrics__employeesrange,
            c.metrics__raised                    AS company__metrics__raised,
            c.metrics__marketcap                 AS company__metrics__marketcap,
            c.geo__city                          AS company__geo__city,
            c.geo__country                       AS company__geo__country,
            c.geo__state                         AS company__geo__state,
            c.tech                               AS company__tech,
            c.tags                               AS company__tags,
            c.founded_year                       AS company__founded_year,
            c.emailprovider                      AS company__emailprovider,
            c.site_url                           AS company__site_url,
            c.alexa__globalrank                  AS company__alexa__globalrank,
            c.is_personal                        AS company__is_personal,
            current_timestamp()                  AS updated_at
        FROM {SILVER_CATALOG}.{SILVER_SCHEMA}.contacts_all ca
        LEFT JOIN {GOLD_CATALOG}.{GOLD_SCHEMA}.persons p
            ON lower(ca.email) = lower(p.mk_email)
        LEFT JOIN {GOLD_CATALOG}.{GOLD_SCHEMA}.companies c
            ON lower(ca.domain) = lower(c.mk_domain)
    """)

    compute_count = df_contacts_computations.count()
    print(f"  contacts_computations rows: {compute_count}")
    df_contacts_computations.show(20, truncate=True)

    DeltaTable.forName(spark, f"{GOLD_CATALOG}.{GOLD_SCHEMA}.contacts_computations").alias("target").merge(
        df_contacts_computations.alias("source"),
        "target.contact_id = source.contact_id AND target.tenant = source.tenant"
    ).whenMatchedUpdateAll().whenNotMatchedInsertAll().execute()

    spark.table(f"{GOLD_CATALOG}.{GOLD_SCHEMA}.contacts_computations").show(20, truncate=True)
    print(f"STEP 3 COMPLETE — {compute_count} rows written to hgi.gold.contacts_computations")

    record_count("3_contacts_computations", f"{GOLD_CATALOG}.{GOLD_SCHEMA}.contacts_computations", compute_count)  # CDF_CHECK
    print(f"  [CDF_CHECK] Step 3 contacts_computations = {compute_count}")                                         # CDF_CHECK

    status = "success"

except Exception as e:
    status       = "failure"
    error_type   = type(e).__name__
    error_reason = str(e)
    print(f"\nERROR: {error_type} — {error_reason}")
    raise



finally:
    end_time    = time.time()
    duration_ms = round((end_time - start_time) * 1000, 2)

    # -----------------------------------------------------------------------
    # CDF ENABLEMENT
    # -----------------------------------------------------------------------
    print("\n" + "="*70)
    print("CDF ENABLEMENT — Enabling Change Data Feed on gold tables")
    print("="*70)

    try:
        spark.sql(f"ALTER TABLE {GOLD_CATALOG}.{GOLD_SCHEMA}.persons SET TBLPROPERTIES ('delta.enableChangeDataFeed' = 'true')")
        spark.sql(f"ALTER TABLE {GOLD_CATALOG}.{GOLD_SCHEMA}.companies SET TBLPROPERTIES ('delta.enableChangeDataFeed' = 'true')")
        spark.sql(f"ALTER TABLE {GOLD_CATALOG}.{GOLD_SCHEMA}.contacts_computations SET TBLPROPERTIES ('delta.enableChangeDataFeed' = 'true')")
        print("  CDF enabled: persons, companies, contacts_computations")
    except Exception as cdf_e:
        print(f"  WARNING: CDF enablement failed: {str(cdf_e)}")

    # -----------------------------------------------------------------------
    # MODULE 2 — CDC VALIDATION (END)
    # Verifies CDF actually enabled on gold tables after writing.
    # Writes results to hgi.gold.logs.cdc_validation_log
    # -----------------------------------------------------------------------
    print("\n" + "="*70)
    print("CDC VALIDATION — Verifying CDF on gold tables")
    print("="*70)

    try:
        cdf_gold_result = validate_cdc(
            layer       = "gold",
            table_names = ["persons", "companies", "contacts_computations"],
            catalog     = GOLD_CATALOG,
            schema      = GOLD_SCHEMA
        )
        cdf_validation_status = "ENABLED_ON_ALL" if cdf_gold_result["status"] else f"FAILED_ON: {cdf_gold_result['disabled_tables']}"
        print(f"  CDC Gold: {cdf_gold_result['message']}")
    except Exception as cdc_e:
        cdf_validation_status = f"ERROR: {str(cdc_e)}"
        print(f"  WARNING: CDC validation failed: {str(cdc_e)}")

    # -----------------------------------------------------------------------
    # CDF_CHECK — WRITE ALL COUNTS TO bronze.cdf_checks.cdf_checker
    # -----------------------------------------------------------------------
    try:                                                                                              # CDF_CHECK
        if cdf_counts:                                                                               # CDF_CHECK
            df_cdf = spark.createDataFrame(cdf_counts).withColumn("run_timestamp", current_timestamp())  # CDF_CHECK
            df_cdf.write.format("delta").mode("append").saveAsTable("bronze.cdf_checks.cdf_checker") # CDF_CHECK
            print(f"  [CDF_CHECK] {len(cdf_counts)} count records written to bronze.cdf_checks.cdf_checker")  # CDF_CHECK
    except Exception as cdf_e:                                                                       # CDF_CHECK
        print(f"  [CDF_CHECK] WARNING: Write failed: {str(cdf_e)}")                                 # CDF_CHECK

    # -----------------------------------------------------------------------
    # MODULE 1 — METRICS QUERY (END)
    # -----------------------------------------------------------------------
    print("\n" + "="*70)
    print("METRICS — Saving pipeline stats")
    print("="*70)

    try:
        persons_rows      = spark.sql(f"SELECT COUNT(*) AS cnt FROM {GOLD_CATALOG}.{GOLD_SCHEMA}.persons").collect()[0]["cnt"]
        companies_rows    = spark.sql(f"SELECT COUNT(*) AS cnt FROM {GOLD_CATALOG}.{GOLD_SCHEMA}.companies").collect()[0]["cnt"]
        computations_rows = spark.sql(f"SELECT COUNT(*) AS cnt FROM {GOLD_CATALOG}.{GOLD_SCHEMA}.contacts_computations").collect()[0]["cnt"]

        print(f"  hgi.gold.persons              : {persons_rows} rows")
        print(f"  hgi.gold.companies            : {companies_rows} rows")
        print(f"  hgi.gold.contacts_computations: {computations_rows} rows")

        total_rows = persons_rows + companies_rows + computations_rows
        mq.save_stats(days=30, rows_processed=total_rows)
        print(f"  Metrics saved | total rows_processed: {total_rows}")
    except Exception as mq_e:
        print(f"  WARNING: Metrics save failed: {str(mq_e)}")

    # -----------------------------------------------------------------------
    # MODULE 3 — AUDIT LOGGER (END) — ACTIVE
    # -----------------------------------------------------------------------
    print("\n" + "="*70)
    print("AUDIT LOGGER — Logging job execution")
    print("="*70)

    try:
        log_audit(
            job_name         = "03_Load_Silver_to_Gold_Augment",
            customer_id      = customer_id,
            status           = status,
            alert_type       = "SUCCESS" if status == "success" else "FAILURE",
            error_type       = error_type,
            error_reason     = error_reason,
            layer            = "gold",
            rows_processed   = total_rows,
            duration_ms      = int(duration_ms),
            email_on_failure = "pratibha.j.kumari@v4c.ai"
        )
    except Exception as al_e:
        print(f"  WARNING: Audit log failed: {str(al_e)}")

    print(f"""
=======================================================================
  AUGMENT LAYER COMPLETE
  Customer : {customer_id}
  Status   : {status.upper()}
=======================================================================
  STEP 0A : {email_count} email candidates
  STEP 0B : {domain_count} domain candidates
  STEP 1  : {persons_enriched_count} persons  -> hgi.gold.persons
  STEP 2  : {companies_enriched_count} companies -> hgi.gold.companies
  STEP 3  : {compute_count} rows   -> hgi.gold.contacts_computations
  Duration: {duration_ms} ms
  CDC     : {cdf_validation_status}
=======================================================================
"""
)